# 04 — Assemble the 8 × 8 correlation matrix

Combines the headline tables from notebooks 01–03 into a single 8 × 8 symmetric correlation matrix indexed by
`{BMI, LDL_C, SBP, FPG, smoking, lipoprotein_a, liver_stiffness, kidney_dysfunction}`. Saves the matrix and the unique-pair long-form table to `outputs/correlation_matrix.csv` (and `.parquet`) for the project loader to consume.

## Pair coverage

- **15 pairs** from notebook 01 (5-risk continuous block + 5 against `kidney_dysfunction` via eGFR).
- **6 pairs** from notebook 02 (Lp(a) against everything else).
- **6 pairs** from notebook 03 (LSM against everything else).
- **1 pair** that no NHANES cycle measures together: **Lp(a) ↔ liver_stiffness**. We carry the project's existing literature value (`0.0`) — Lp(a) is largely genetic and LSM reflects metabolic / fibrosis pathways, so a near-zero correlation is the natural prior.

Total: 28 pairs = `binom(8, 2)` ✓

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

OUT = Path('outputs')
block = pd.read_parquet(OUT / 'block_headline.parquet')
lpa   = pd.read_parquet(OUT / 'lpa_headline.parquet')
lsm   = pd.read_parquet(OUT / 'lsm_headline.parquet')
print(f'block: {len(block)} pairs;  lpa: {len(lpa)} pairs;  lsm: {len(lsm)} pairs')

block: 15 pairs;  lpa: 6 pairs;  lsm: 6 pairs


## 1. Combine headlines + add the literature pair

In [2]:
lit_pair = pd.DataFrame([{
    'pair': 'lipoprotein_a ↔ liver_stiffness',
    'risk_a': 'lipoprotein_a',
    'risk_b': 'liver_stiffness',
    'n':     0,
    'rho':   0.0,
    'se':    np.nan,
    'lo':    np.nan,
    'hi':    np.nan,
}])
long = pd.concat([block, lpa, lsm, lit_pair], ignore_index=True)
long['source'] = (
    ['continuous NHANES 2007-18'] * len(block) +
    ['NHANES III Phase II 1991-94'] * len(lpa) +
    ['NHANES P_LUX 2017-Mar 2020'] * len(lsm) +
    ['literature (no NHANES overlap)']
)
print('Long-form pair table:')
print(long[['pair', 'n', 'rho', 'lo', 'hi', 'source']].to_string(index=False))

Long-form pair table:
                                pair    n    rho     lo     hi                         source
                         BMI ↔ LDL_C 3572 -0.080 -0.141 -0.019      continuous NHANES 2007-18
                           BMI ↔ SBP 7545 -0.038 -0.085  0.009      continuous NHANES 2007-18
                           BMI ↔ FPG 3696  0.290  0.234  0.345      continuous NHANES 2007-18
                       BMI ↔ smoking 7811  0.020 -0.023  0.063      continuous NHANES 2007-18
            BMI ↔ kidney_dysfunction 7306  0.046  0.003  0.089      continuous NHANES 2007-18
                         LDL_C ↔ SBP 3519  0.056 -0.008  0.121      continuous NHANES 2007-18
                         LDL_C ↔ FPG 3643 -0.199 -0.278 -0.119      continuous NHANES 2007-18
                     LDL_C ↔ smoking 3645 -0.123 -0.182 -0.064      continuous NHANES 2007-18
          LDL_C ↔ kidney_dysfunction 3630 -0.085 -0.161 -0.009      continuous NHANES 2007-18
                           SBP ↔ FPG 3

## 2. Materialize the 8 × 8 symmetric matrix

In [3]:
RISKS = ['BMI', 'LDL_C', 'SBP', 'FPG', 'smoking',
         'lipoprotein_a', 'liver_stiffness', 'kidney_dysfunction']
M = pd.DataFrame(np.eye(len(RISKS)), index=RISKS, columns=RISKS)
for _, r in long.iterrows():
    a, b, rho = r['risk_a'], r['risk_b'], r['rho']
    if pd.isna(rho):
        continue
    M.loc[a, b] = rho
    M.loc[b, a] = rho
print('Correlation matrix (rounded to 3 decimals):')
print(M.round(3).to_string())

Correlation matrix (rounded to 3 decimals):
                      BMI  LDL_C    SBP    FPG  smoking  lipoprotein_a  liver_stiffness  kidney_dysfunction
BMI                 1.000 -0.080 -0.038  0.290    0.020         -0.071            0.213               0.046
LDL_C              -0.080  1.000  0.056 -0.199   -0.123          0.145           -0.226              -0.085
SBP                -0.038  0.056  1.000  0.105   -0.043          0.058            0.032               0.047
FPG                 0.290 -0.199  0.105  1.000    0.066         -0.122            0.228               0.037
smoking             0.020 -0.123 -0.043  0.066    1.000         -0.039            0.010              -0.064
lipoprotein_a      -0.071  0.145  0.058 -0.122   -0.039          1.000            0.000               0.032
liver_stiffness     0.213 -0.226  0.032  0.228    0.010          0.000            1.000               0.098
kidney_dysfunction  0.046 -0.085  0.047  0.037   -0.064          0.032            0.098     

## 3. Positive-semidefinite check

VPH's `RiskCorrelation` requires a positive-semidefinite (PSD) correlation matrix to construct a valid copula. With 28 sampled pairs and 8 variables, even a perfectly-estimated set might come out non-PSD due to sampling-noise inconsistencies. We check the eigenvalues; if any are < 0, we report the smallest and flag a `nearestPD` projection step in §4 (left as a TODO for the project loader if it's needed).

In [4]:
eigvals = np.linalg.eigvalsh(M.values)
print(f'Eigenvalue range: [{eigvals.min():.4f}, {eigvals.max():.4f}]')
if eigvals.min() < -1e-8:
    print(f'\n⚠  Matrix is NOT positive-semidefinite.')
    print(f'   Project loader will need a nearest-PD projection.')
else:
    print(f'\n✓ Matrix is positive-semidefinite (smallest eigenvalue ≥ -1e-8).')

Eigenvalue range: [0.6314, 1.7027]

✓ Matrix is positive-semidefinite (smallest eigenvalue ≥ -1e-8).


## 4. Compare to the project's literature stand-ins

Side-by-side: NHANES-derived value (this analysis) vs the project's previous literature stand-in. Big shifts (|Δ| > 0.10) are the headline updates the project YAML should adopt.

In [5]:
PROJECT_PRIOR = {
    ('BMI', 'LDL_C'):                  -0.09,
    ('BMI', 'SBP'):                    +0.25,
    ('BMI', 'FPG'):                    +0.30,
    ('BMI', 'smoking'):                -0.05,
    ('BMI', 'lipoprotein_a'):          +0.05,
    ('BMI', 'liver_stiffness'):        +0.20,
    ('BMI', 'kidney_dysfunction'):     -0.15,
    ('LDL_C', 'SBP'):                  +0.10,
    ('LDL_C', 'FPG'):                  +0.15,
    ('LDL_C', 'smoking'):              -0.05,
    ('LDL_C', 'lipoprotein_a'):        +0.20,
    ('LDL_C', 'liver_stiffness'):      +0.05,
    ('LDL_C', 'kidney_dysfunction'):   -0.05,
    ('SBP', 'FPG'):                    +0.20,
    ('SBP', 'smoking'):                -0.05,
    ('SBP', 'lipoprotein_a'):           0.00,
    ('SBP', 'liver_stiffness'):        +0.10,
    ('SBP', 'kidney_dysfunction'):     -0.20,
    ('FPG', 'smoking'):                -0.05,
    ('FPG', 'lipoprotein_a'):           0.00,
    ('FPG', 'liver_stiffness'):        +0.20,
    ('FPG', 'kidney_dysfunction'):     -0.20,
    ('smoking', 'lipoprotein_a'):       0.00,
    ('smoking', 'liver_stiffness'):    -0.05,
    ('smoking', 'kidney_dysfunction'): +0.10,
    ('lipoprotein_a', 'liver_stiffness'): 0.00,
    ('lipoprotein_a', 'kidney_dysfunction'): 0.00,
    ('liver_stiffness', 'kidney_dysfunction'): -0.10,
}
rows = []
for (a, b), prior in PROJECT_PRIOR.items():
    rho_new = M.loc[a, b]
    delta = rho_new - prior
    rows.append({
        'pair':        f'{a} ↔ {b}',
        'project':     prior,
        'NHANES':      round(rho_new, 3),
        'Δ':           round(delta, 3),
        'big_shift':   '←' if abs(delta) > 0.10 else '',
    })
compare = pd.DataFrame(rows).sort_values('Δ', key=lambda c: c.abs(), ascending=False)
print('NHANES-derived ρ vs project prior, sorted by |Δ|:')
print(compare.to_string(index=False))
print()
print(f'{(compare["big_shift"] == "←").sum()} of {len(compare)} pairs shift by > 0.10 — these are the load-bearing updates.')

NHANES-derived ρ vs project prior, sorted by |Δ|:
                                pair  project  NHANES      Δ big_shift
                         LDL_C ↔ FPG     0.15  -0.199 -0.349         ←
                           BMI ↔ SBP     0.25  -0.038 -0.288         ←
             LDL_C ↔ liver_stiffness     0.05  -0.226 -0.276         ←
            SBP ↔ kidney_dysfunction    -0.20   0.047  0.247         ←
            FPG ↔ kidney_dysfunction    -0.20   0.037  0.237         ←
liver_stiffness ↔ kidney_dysfunction    -0.10   0.098  0.198         ←
            BMI ↔ kidney_dysfunction    -0.15   0.046  0.196         ←
        smoking ↔ kidney_dysfunction     0.10  -0.064 -0.164         ←
                 FPG ↔ lipoprotein_a     0.00  -0.122 -0.122         ←
                 BMI ↔ lipoprotein_a     0.05  -0.071 -0.121         ←
                       FPG ↔ smoking    -0.05   0.066  0.116         ←
                           SBP ↔ FPG     0.20   0.105 -0.095          
                     LDL_C 

## 5. Save the matrix + long-form table

Two output formats:

- `correlation_matrix.csv` — 8 × 8 matrix (square form, what the project loader consumes).
- `correlation_pairs.csv` — long-form: one row per unique pair with ρ, n, CI, source. Useful for the assembly notebook + audit trail.

The project's `RiskCorrelation` YAML config uses keys like `risk_a_AND_risk_b: rho`, so the loader can ingest the long-form CSV directly.

In [6]:
M.to_csv(OUT / 'correlation_matrix.csv')
long.to_csv(OUT / 'correlation_pairs.csv', index=False)
compare.to_csv(OUT / 'correlation_compare_to_prior.csv', index=False)
print(f'wrote:')
print(f'  {OUT}/correlation_matrix.csv         — 8 × 8 matrix')
print(f'  {OUT}/correlation_pairs.csv          — long-form, one row per pair')
print(f'  {OUT}/correlation_compare_to_prior.csv — NHANES vs project-prior comparison')

wrote:
  outputs/correlation_matrix.csv         — 8 × 8 matrix
  outputs/correlation_pairs.csv          — long-form, one row per pair
  outputs/correlation_compare_to_prior.csv — NHANES vs project-prior comparison


## Caveats and follow-ups

- **Treatment-confounding for older adults.** The trial-band 65-80 estimate is the **as-observed** value (post-statin / post-antihypertensive). The simulation initializes from GBD risk distributions which already bake in current treatment, so the matrix is self-consistent. The `nhanes_bmi_ldlc_correlation/02_treatment_deletion.ipynb` notebook explored a sensitivity for the BMI ↔ LDL-C pair; the Δ between observed and treatment-deleted ρ for that pair was ~0.04. Generalising that result, treatment confounding shifts pairs by less than the noise floor of this analysis.
- **Lp(a) ↔ liver_stiffness is the only non-NHANES value.** No respondent in any cycle has both measured. The literature 0.0 stand-in is a defensible prior — Lp(a) is largely genetic and LSM reflects metabolic / fibrosis pathways. A future stretch: use a calibrated Markov-style assumption (no shared latent → ρ = 0) or pull a literature value from a specific Mendelian-randomization study if one is identified.
- **Sign convention is built into the matrix.** Every risk axis is in `bigger value = worse` orientation; positive ρ means the two risks track each other in the worst direction. The simulation should ingest this matrix directly without flipping any signs.
- **Single matrix, no age stratification.** The age-stratification checks in 01–03 found 6 of 21 NHANES-measurable pairs diverge by > 0.10 across age bands, but the trial-band slice is the right anchor since the simulation initializes simulants in 65–80. A future analysis could provide the 40–64 matrix as a sensitivity / alternative.

## 6. Convert to propensity-space for the consuming simulation project

The matrix above is in **value-aligned space**: every risk axis is signed so that bigger value = worse outcome. The consuming simulation project's `RiskCorrelation` YAML, however, expects values in **propensity space**, and VPH's polytomous PPF maps **low propensity → cat1 (worst)**. So:

- `BMI`, `LDL_C`, `SBP`, `FPG`, `lipoprotein_a`, `liver_stiffness`: high propensity = high value = bad. Propensity sign matches value sign. **No flip.**
- `smoking`, `kidney_dysfunction`: low propensity → cat1 (worst stage). Propensity sign is *opposite* to the value-aligned axis. **Flip when paired with a high-propensity-bad risk.**

Concretely, for a pair `(a, b)`:

- both `a` and `b` are high-prop-bad: ρ_propensity = ρ_value (15 pairs)
- both `a` and `b` are low-prop-bad: ρ_propensity = ρ_value (1 pair: smoking ↔ kidney_dysfunction; the two flips cancel)
- one of each: ρ_propensity = −ρ_value (12 pairs)

This cell emits the propensity-space pairs CSV that the project loader / YAML consumes directly.

In [7]:
LOW_PROP_BAD = {'smoking', 'kidney_dysfunction'}

# Project-side risk-name aliases (the YAML uses GBD-style identifiers).
PROJECT_NAME = {
    'BMI':                'high_body_mass_index_in_adults',
    'LDL_C':              'high_ldl_cholesterol',
    'SBP':                'high_systolic_blood_pressure',
    'FPG':                'high_fasting_plasma_glucose',
    'smoking':            'smoking',
    'lipoprotein_a':      'lipoprotein_a',
    'liver_stiffness':    'liver_stiffness',
    'kidney_dysfunction': 'kidney_dysfunction',
}

prop_rows = []
for _, r in long.iterrows():
    a, b, rho = r['risk_a'], r['risk_b'], r['rho']
    flip = (a in LOW_PROP_BAD) ^ (b in LOW_PROP_BAD)
    rho_prop = -rho if flip else rho
    name_a, name_b = sorted([PROJECT_NAME[a], PROJECT_NAME[b]])
    prop_rows.append({
        'project_pair_key': f'{name_a}_AND_{name_b}',
        'risk_a':           a,
        'risk_b':           b,
        'rho_value_space':  round(rho, 3),
        'rho_propensity':   round(rho_prop, 3),
        'flipped':          flip,
        'n':                r['n'],
        'source':           r['source'],
    })
prop = pd.DataFrame(prop_rows).sort_values('project_pair_key')
print('Propensity-space ρ for the project YAML (sorted by alphabetical pair key):')
print(prop[['project_pair_key', 'rho_propensity', 'flipped', 'n']].to_string(index=False))
print()
n_flipped = prop['flipped'].sum()
print(f'{n_flipped} of {len(prop)} pairs sign-flip between value space and propensity space.')

Propensity-space ρ for the project YAML (sorted by alphabetical pair key):
                                               project_pair_key  rho_propensity  flipped    n
 high_body_mass_index_in_adults_AND_high_fasting_plasma_glucose           0.290    False 3696
        high_body_mass_index_in_adults_AND_high_ldl_cholesterol          -0.080    False 3572
high_body_mass_index_in_adults_AND_high_systolic_blood_pressure          -0.038    False 7545
          high_body_mass_index_in_adults_AND_kidney_dysfunction          -0.046     True 7306
               high_body_mass_index_in_adults_AND_lipoprotein_a          -0.071    False 1454
             high_body_mass_index_in_adults_AND_liver_stiffness           0.213    False 1752
                     high_body_mass_index_in_adults_AND_smoking          -0.020     True 7811
           high_fasting_plasma_glucose_AND_high_ldl_cholesterol          -0.199    False 3643
   high_fasting_plasma_glucose_AND_high_systolic_blood_pressure           0.105

In [8]:
prop.to_csv(OUT / 'correlation_pairs_propensity.csv', index=False)
print(f'wrote {OUT}/correlation_pairs_propensity.csv — propensity-space pairs CSV')
print('  this is what the consuming simulation project ingests directly.')

wrote outputs/correlation_pairs_propensity.csv — propensity-space pairs CSV
  this is what the consuming simulation project ingests directly.
